<a href="https://colab.research.google.com/github/Ben-m13/SCA_LocationOptimization/blob/main/SCA_HW2_LocationOptimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SCA HW2 - Parcel Pick-up Facility Location Optimization

We utilized the Location Optimization code from Dr. Osadchiy and modified it for finding the optimal location in Atlanta for a Parcel Pick-up Facility using the 10 most populous [Atlanta neighborhoods](https://en.wikipedia.org/wiki/Table_of_Atlanta_neighborhoods_by_population). Neighborhood population was used for the weighting of the distance calculations, and a finer grid was used as part of the grid search for driving distance optimal location.

In [ ]:
!pip install googlemaps
!pip install polyline
!pip install folium

import pandas as pd
import math
import scipy.optimize as opt
from geopy.geocoders import GoogleV3
import googlemaps
import polyline
import folium

# Enter your own API key
geolocator = GoogleV3(api_key='')
gmaps = googlemaps.Client(key='')    # Initialize the Google Maps client with your own API key


  Preparing metadata (setup.py) ... done
  Created wheel for googlemaps: filename=googlemaps-4.10.0-py3-none-any.whl size=40714 sha256=9e71cadb1024c16cde7bee7e025d76eeec2f0080dd2fd91d33cb3e222058f0a0
  Stored in directory: /root/.cache/pip/wheels/4c/6a/a7/bbc6f5c200032025ee655deb5e163ce8594fa05e67d973aad6
Successfully built googlemaps


In [ ]:
# Define functions

# Function to get geocode for a location
def get_geocode(location):
    coords = geolocator.geocode(location)
    lat = round(coords.latitude, 4)
    lng = round(coords.longitude, 4)
    return lat, lng

# Great circle ("as crow flies") distance
def calc_dist_haversine(lat1, lng1, lat2, lng2):
  # Convert latitude and longitude from degrees to radians
  lat1, lng1, lat2, lng2 = map(math.radians, [lat1, lng1, lat2, lng2])

  a = math.sin((lat2 - lat1) / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin((lng2 - lng1) / 2) ** 2
  dist_haversine_miles = 3959 * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

  return dist_haversine_miles

# Function to calculate Haversine distance from a point to locations in dataframe
def calc_cost_haversine(coords, data, weight_col="Weight"):
    lat, lon = coords
    distances = []
    for _, row in data.iterrows():
        distances.append(calc_dist_haversine(lat, lon, row["Lat"], row["Lng"]))

    w = data[weight_col].astype(float)
    return (w * pd.Series(distances, index=data.index)).sum()

# Function to calculate travel distance from origin to each destination
def calc_dist_driving(lat1, lng1, lat2, lng2):

    # Request directions and get travel distance and travel time
    directions = gmaps.directions( (lat1, lng1), (lat2, lng2), mode='driving', units='imperial')

    dist_travel_mile = directions[0]['legs'][0]['distance']['value'] / 1609.344

    return dist_travel_mile

# Function to calculate travel distance from a point to locations in dataframe
def calc_cost_driving(coords, data, weight_col="Weight"):
    lat, lon = coords
    distances = []
    for _, row in data.iterrows():
        distances.append(calc_dist_driving(lat, lon, row["Lat"], row["Lng"]))

    w = data[weight_col].astype(float)
    return (w * pd.Series(distances, index=data.index)).sum()


In [ ]:
# Creating the DataFrame
df = pd.DataFrame({
    'Location': ['Midtown, Atlanta, GA','Downtown, Atlanta, GA','Old Fourth Ward, Atlanta, GA','North Buckhead, Atlanta, GA',
                 'Pine Hills, Atlanta, GA','Morningside, Atlanta, GA','Virginia-Highland, Atlanta, GA','Grant Park, Atlanta, GA','Georgia Tech, Atlanta, GA','Kirkwood, Atlanta, GA'],
    'Population': [16569,13411,10505,8270,8033,8030,7800,6771,6607,5897]})

df["Weight"] = df["Population"].astype(float)


# Get geocodes for each address in the DataFrame
df[['Lat', 'Lng']] = df['Location'].apply(lambda x: pd.Series(get_geocode(x)))

df

,Location,Population,Weight,Lat,Lng
0,"Midtown, Atlanta, GA",16569,16569.0,33.7833,-84.3831
1,"Downtown, Atlanta, GA",13411,13411.0,33.7557,-84.3884
2,"Old Fourth Ward, Atlanta, GA",10505,10505.0,33.7640,-84.3720
3,"North Buckhead, Atlanta, GA",8270,8270.0,33.8527,-84.3654
4,"Pine Hills, Atlanta, GA",8033,8033.0,33.8375,-84.3516
5,"Morningside, Atlanta, GA",8030,8030.0,33.7962,-84.3595
6,"Virginia-Highland, Atlanta, GA",7800,7800.0,33.7817,-84.3635
7,"Grant Park, Atlanta, GA",6771,6771.0,33.7357,-84.3712
8,"Georgia Tech, Atlanta, GA",6607,6607.0,33.7780,-84.3980
9,"Kirkwood, Atlanta, GA",5897,5897.0,33.7533,-84.3262


## Haversine Distance Optimal Location

In [ ]:
# Initial guess for the optimization algorithm (somewhere close to the average of latitudes and longitudes)
initial_guess = [df['Lat'].mean(), df['Lng'].mean()]

# Extract min and max values of Latitude and Longitude columns for bounds
bounds = [(df['Lat'].min() - 5, df['Lat'].max() + 5), (df['Lng'].min() - 5, df['Lng'].max() + 5)]

# Minimize the total distance function using scipy's minimize
result = opt.minimize(calc_cost_haversine, initial_guess, args=(df,), method='SLSQP', bounds=bounds)

# Get the location address
result_address = gmaps.reverse_geocode((result.x))[0]['formatted_address']

print(f"\nHaversine Distance Optimal Location (lat, long): \n{(result.x).round(4) }\nAddress: {result_address}")


Haversine Distance Optimal Location (lat, long): 
[ 33.7838 -84.3679]
Address: 1071 Monroe Dr NE, Atlanta, GA 30306, USA


## Driving Distance Optimal Location

In [ ]:
# Define ranges based on optimal point ± 0.25 (Grid Search)

# Modified from original code for smaller difference, and use step to denote
# movement within the grid
diff = 0.25
step = 0.02

ranges = (
    slice(result.x[0] - diff, result.x[0] + diff + step, step),
    slice(result.x[1] - diff, result.x[1] + diff + step, step)
)

# Minimize the total distance function using scipy's integer minimizer
result_driving = opt.brute(calc_cost_driving, ranges, args=(df,), full_output=True, finish=None)

# Get the location address
result_driving_address = gmaps.reverse_geocode((result_driving[0]))[0]['formatted_address']

print(f"\nDriving Distance Optimal Location (lat, long): \n{ result_driving[0].tolist() }\nAddress: {result_driving_address}")




Driving Distance Optimal Location (lat, long): 
[33.773810000000005, -84.37789000000001]
Address: 708 Argonne Ave NE, Atlanta, GA 30308, USA


In [ ]:
### Code to print locations on map

# Initialize the map centered around the mean of all locations
m = folium.Map(location = initial_guess, zoom_start=5)

# Add markers for optimal locations

# Point using Haversine distnace
folium.Marker(
    location=result.x,
    tooltip="Haversine Distance Optimal Location",
    popup=f"{result_address}",
    icon=folium.Icon(color='orange')).add_to(m)

# Point using Driving distance
folium.Marker(
    location=result_driving[0],
    tooltip="Driving Distance Optimal Location",
    popup=f"{result_driving_address}",
    icon=folium.Icon(color='green')).add_to(m)

# Draw driving paths between locations
for i in range(len(df['Location'])):
    start = tuple(result_driving[0])
    end   = (df['Lat'][i], df['Lng'][i])

    # Add markers for locations from DataFrame
    folium.Marker([df['Lat'][i], df['Lng'][i]], popup=df['Location'][i]).add_to(m)
    # Get directions using Google Maps Directions API from optimal point to all locations
    directions = gmaps.directions(start, end, mode="driving")
    # Extract polyline points from API response and add to Folium map
    points = polyline.decode(directions[0]['overview_polyline']['points'])
    # Add driving path on the map
    folium.PolyLine(locations=points, color='blue', weight=5).add_to(m)


## Final Results (Map of Optimal Locations Included)

In [ ]:
display(m)
# Orange = Haversine Distance Optimal
# Green = Driving Distance Optimal
# Blue = Neighborhood Locations

print(f"\nHaversine Distance Optimal Location (lat, long): \n{(result.x).round(4) }\nAddress: {result_address}")
print(f"\nDriving Distance Optimal Location (lat, long): \n{ result_driving[0].tolist() }\nAddress: {result_driving_address}")



Haversine Distance Optimal Location (lat, long): 
[ 33.7838 -84.3679]
Address: 1071 Monroe Dr NE, Atlanta, GA 30306, USA

Driving Distance Optimal Location (lat, long): 
[33.773810000000005, -84.37789000000001]
Address: 708 Argonne Ave NE, Atlanta, GA 30308, USA
